# RetailIQ Analytics — SQL Business Analysis

This notebook performs business analysis using PostgreSQL on the cleaned RetailIQ e-commerce dataset.

The objective is to answer important business questions using SQL and convert the results into actionable business insights.

## Analysis Framework

Each analysis follows:

**Business Question → SQL Query → Result → Insight → Business Recommendation**

## Key Analysis Areas

- Revenue and sales performance
- Order and customer behavior
- Product and category performance
- Seller performance
- Payment behavior
- Delivery performance
- Customer retention
- Repeat purchase behavior
- Customer value

In [ ]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="localhost",
    database="retailiq",
    user="postgres",
    password="password"
)

print("PostgreSQL connection successful!")

PostgreSQL connection successful!


## Database Connection

A connection is established between Python and the PostgreSQL `retailiq` database.

Python is used to execute SQL queries and retrieve the results into pandas DataFrames for analysis.

The database contains seven core analytical tables:

- customers
- orders
- order_items
- order_payments
- order_reviews
- products
- sellers

## 1. Total Revenue

### Business Question

How much revenue has the business generated from the available order data?

### Approach

Revenue is calculated from the `order_items` table using the `price` column.

Since an order can contain multiple items, revenue must be aggregated across all order items.

### Result

The total item-level revenue generated across all orders is **13,591,643.70**.

### Business Insight

The `order_items` table provides the item-level transaction values required to calculate total sales revenue.

This metric serves as the baseline revenue measure for subsequent analyses such as monthly revenue trends, category performance, seller performance, and average order value.

## 2. Total Number of Orders

### Business Question

How many orders were placed in the dataset?

### SQL Approach

Each row in the `orders` table represents one order, so the total number of orders can be calculated by counting `order_id`.

## 3. Total Unique Customers

### Business Question

How many unique customers have placed orders?

### SQL Approach

The `orders` table can contain multiple orders from the same customer.

Therefore, counting rows is not enough to determine the number of unique customers.

We use `COUNT(DISTINCT customer_id)` to count each customer only once.

### Result

The `orders` table contains 99,441 unique customer records.

However, `customer_id` represents the customer record associated with an order. For repeat-customer analysis, the more appropriate identifier is `customer_unique_id` from the `customers` table.

Therefore, customer retention and repeat-purchase analysis will be performed using `customer_unique_id`.

## 4. Average Order Value (AOV)

### Business Question

What is the average revenue generated per order?

### Formula

**AOV = Total Revenue ÷ Total Orders**

AOV helps measure the average monetary value of an order and is useful for evaluating customer spending and sales performance.

### Result

The Average Order Value (AOV) is **137.75**.

This means the average order generated approximately 137.75 in item-level revenue.

The AOV was calculated using distinct orders because an order can contain multiple items.

### Business Insight

AOV provides a baseline measure of customer spending per order and can be used to compare customer segments, product categories, payment methods, and time periods.

## 5. Total Items Sold

### Business Question

How many individual product items were sold across all orders?

### SQL Approach

Each row in `order_items` represents one item within an order. Therefore, counting the rows gives the total number of items sold.

## 6. Average Items per Order

### Business Question

How many product items does the average order contain?

### Formula

**Average Items per Order = Total Items Sold ÷ Total Orders**

This metric helps understand the typical number of items included in each order.

### Result

The average order contains approximately **1.14 items**.

This indicates that most orders contain a single product item, while a smaller proportion of orders contain multiple items.

### Business Insight

The relatively low average items per order suggests that increasing the number of items purchased per order could be one potential way to increase order value, for example through product bundles or cross-selling.

## 7. Revenue by Order Status

### Business Question

How is item-level revenue distributed across different order statuses?

### SQL Approach

The `orders` table contains the order status, while revenue is stored in `order_items`.

Therefore, the two tables need to be joined using `order_id`.

Revenue is then grouped by `order_status`.

### Result

Revenue is heavily concentrated in orders with a `delivered` status.

Delivered-status orders account for approximately 97.3% of the total item-level revenue, while the remaining revenue is associated with shipped, canceled, invoiced, processing, unavailable, and approved orders.

### Business Insight

Most of the dataset's item-level revenue is associated with successfully delivered orders.

However, revenue associated with canceled or other non-delivered orders should not automatically be treated as realized revenue. Further analysis of order status and payment data is required to understand the actual financial impact.

## 8. Monthly Revenue Trend

### Business Question

How does revenue change month by month?

### SQL Approach

The order purchase timestamp is stored in the `orders` table, while revenue is stored in `order_items`.

We join the two tables using `order_id`, extract the month from the purchase timestamp, and aggregate item-level revenue by month.

### Result

Monthly revenue shows a strong upward trend during 2017, with revenue increasing substantially compared with the early months of the dataset.

November 2017 recorded the highest monthly revenue at approximately **1.01 million**, while December revenue decreased to approximately **743.91K**.

The first and last months of the dataset may represent partial periods, so they should be interpreted carefully when comparing monthly performance.

### Business Insight

The monthly revenue trend indicates that sales activity increased significantly during 2017, with particularly strong performance toward the later part of the year.

Further analysis should examine monthly order volume and Average Order Value (AOV) separately to determine whether revenue growth was driven primarily by more orders, higher order values, or both.

## 9. Monthly Revenue, Orders and AOV

### Business Question

How do monthly revenue, order volume, and Average Order Value change over time?

### Objective

Revenue alone does not explain why sales changed.

By combining:

- Monthly revenue
- Number of orders
- Average Order Value

we can determine whether revenue changes are driven by order volume, customer spending per order, or both.

### Result

The monthly analysis shows that revenue growth is accompanied by a substantial increase in order volume.

Average Order Value (AOV) remains comparatively stable across most months, generally staying within a relatively narrow range compared with the large changes in monthly order volume.

### Business Insight

The results suggest that changes in revenue are driven substantially by changes in order volume rather than large changes in AOV.

Therefore, future analysis should focus on customer acquisition, order frequency, and retention to understand the drivers of sales growth.

## 10. Monthly Revenue Growth

### Business Question

How much did revenue increase or decrease compared with the previous month?

### SQL Approach

First, monthly revenue is calculated.

Then the `LAG()` window function retrieves the previous month's revenue.

Monthly revenue growth is calculated as:

**Revenue Growth % = (Current Month Revenue - Previous Month Revenue) / Previous Month Revenue × 100**

### Result

Monthly revenue growth shows periods of both expansion and contraction.

Revenue increased by **52.10% in November 2017** compared with October 2017, followed by a **26.36% decline in December 2017**.

The very large percentage changes during the earliest months are influenced by the small amount of revenue recorded in the initial partial periods and should not be interpreted as normal month-over-month growth.

### Business Insight

The November 2017 revenue increase is a notable period for further investigation. The next analysis can break down monthly performance by order volume, product category, and seller to identify the factors associated with the increase.

The `LAG()` window function provides a useful method for comparing each month's performance with the immediately preceding month.

## 11. Revenue by Product Category

### Business Question

Which product categories generate the highest revenue?

### SQL Approach

The `order_items` table contains item prices and product IDs, while the `products` table contains product category information.

The two tables are joined using `product_id`, and revenue is aggregated by product category.

### Result

Revenue is concentrated across several major product categories.

The top revenue-generating categories are:

1. `beleza_saude`
2. `relogios_presentes`
3. `cama_mesa_banho`
4. `esporte_lazer`
5. `informatica_acessorios`

The top categories contribute a substantial portion of total item-level revenue.

### Business Insight

Category-level revenue analysis helps identify the product groups that contribute most to overall sales.

However, revenue alone does not explain category performance. Further analysis should compare revenue share, order volume, Average Order Value (AOV), and customer behavior across categories.

## 12. Top 10 Category Revenue Share

### Business Question

What percentage of total revenue is generated by the top 10 product categories?

### SQL Approach

First, revenue is calculated for each product category.

The top 10 categories are then selected and their combined revenue is compared with total item-level revenue.

### Result

The top 10 product categories contribute **62.36%** of total item-level revenue.

### Business Insight

Revenue is concentrated across a relatively small group of product categories.

This indicates that these categories have a significant influence on overall revenue performance. However, category importance should not be evaluated using revenue alone.

Further analysis should compare category revenue with order volume, Average Order Value (AOV), and customer repeat behavior to understand the underlying drivers of category performance.

## 13. Category Revenue, Orders and AOV

### Business Question

How do product categories compare in terms of revenue, order volume, and Average Order Value?

### Objective

Compare category performance using multiple metrics rather than revenue alone.

Metrics:

- Number of orders
- Revenue
- Average Order Value

### Result

Category performance varies considerably when revenue, order volume, and AOV are considered together.

For example, `beleza_saude` generates the highest revenue among the displayed categories with 8,836 orders and an AOV of 142.45.

`relogios_presentes` generates slightly lower revenue but has a substantially higher AOV of 214.26.

`cama_mesa_banho` has the highest order volume among these top categories at 9,417 orders, while its AOV is lower at 110.12.

The `pcs` category has a very high AOV of 1,231.84, but only 181 orders, so its AOV should be interpreted in the context of its relatively small order volume.

### Business Insight

Category performance should be evaluated using multiple metrics rather than revenue alone. High revenue can result from high order volume, high AOV, or a combination of both.

## 14. Seller Revenue Performance

### Business Question

Which sellers generate the highest item-level revenue?

### SQL Approach

The `order_items` table contains seller IDs and item prices.

The `sellers` table contains seller information.

The tables are joined using `seller_id`, and revenue is aggregated at the seller level.

### Result

Seller revenue varies substantially across sellers.

The top 10 sellers show different combinations of order volume and revenue. For example, one seller generated approximately 222.8K revenue from only 358 orders, while another generated approximately 200.5K revenue from 1,806 orders.

### Business Insight

Seller performance should not be evaluated using total revenue alone.

Comparing seller revenue with order volume can help identify whether a seller's revenue is primarily driven by high order volume or higher-value orders.

The next step is to calculate seller-level AOV to make this comparison more meaningful.

## 15. Seller Revenue and AOV

### Business Question

Which sellers generate high revenue, and what is their average order value?

### Objective

Compare seller revenue with order volume and AOV to understand different seller performance patterns.

### Result

Seller performance differs significantly when revenue, order volume, and AOV are considered together.

For example, seller `5324...` generated approximately 222.8K revenue from 358 orders, resulting in an AOV of 622.28.

In contrast, seller `4a3c...` generated approximately 200.5K revenue from 1,806 orders, with an AOV of 111.00.

### Business Insight

Seller revenue can be driven by either high order volume or higher-value orders.

Therefore, seller performance analysis should consider revenue, order volume, and AOV together rather than relying on revenue alone.

## 16. Payment Method Analysis

### Business Question

Which payment methods are most frequently used, and how much payment value is associated with each method?

### SQL Approach

The `order_payments` table contains individual payment records.

We group the records by `payment_type` and calculate:

- Number of payment records
- Total payment value
- Average payment value

### Result

Credit card is the dominant payment method by total payment value, with approximately **12.54 million** in payment value.

Boleto is the second-largest payment method at approximately **2.87 million**.

Voucher payments have a lower average payment value of approximately **65.70** compared with credit card payments at **163.32**.

### Business Insight

Payment value is heavily concentrated in credit card transactions.

However, `payment_count` represents payment records rather than unique orders because a single order can contain multiple payment records. Therefore, payment-method adoption should be analyzed separately from order-level behavior.

## 17. Payment Method Share

### Business Question

What percentage of total payment value comes from each payment method?

### SQL Approach

Payment value is aggregated by payment method and divided by the total payment value across all payment records.

### Result

Credit card payments account for **78.34%** of total payment value, followed by boleto at **17.92%**.

Together, credit card and boleto account for **96.26%** of total payment value.

### Business Insight

Payment value is highly concentrated in two payment methods.

Credit card is the dominant payment method by payment value, while boleto represents the second-largest contribution. Voucher and debit card contribute comparatively smaller shares of total payment value.

This concentration can be useful when evaluating payment-method performance and potential payment-related business initiatives.

## 18. Customer Order Frequency

### Business Question

How many orders does each unique customer place?

### SQL Approach

The `customer_unique_id` from the `customers` table is used to identify customers across multiple orders.

Orders are grouped by unique customer to calculate customer-level order frequency.

### Result

Customer order frequency varies considerably.

The highest-frequency customer placed 17 orders, while several other customers placed between 5 and 9 orders.

This confirms that the dataset contains customers with multiple purchases.

### Business Insight

Customer behavior is not uniform. While many customers may place only one order, a smaller group places multiple orders and can contribute disproportionately to customer lifetime revenue.

Further analysis will classify customers into one-time and repeat customers to measure the size and revenue contribution of each group.

## 19. One-Time vs Repeat Customers

### Business Question

What proportion of customers are one-time buyers versus repeat buyers?

### Customer Classification

- **One-time customer:** exactly 1 order
- **Repeat customer:** more than 1 order

The analysis uses `customer_unique_id` so that multiple customer records belonging to the same real customer are counted correctly.

### Result

The analysis shows that:

- 93,099 customers (96.88%) are one-time customers.
- 2,997 customers (3.12%) are repeat customers.

### Business Insight

The customer base is heavily weighted toward one-time purchasers, while repeat customers represent a relatively small share of the customer base.

This makes customer retention and repeat-purchase behavior important areas for further investigation.

However, this result alone does not explain why customers do not return. Further analysis of repeat-purchase timing, customer value, product categories, and cohort retention is required.

## 20. Revenue Contribution by Customer Type

### Business Question

How much revenue is generated by one-time versus repeat customers?

### Objective

Compare customer share with revenue contribution to understand the economic contribution of each customer group.

### Result

One-time customers represent **96.88%** of the customer base and contribute **94.27%** of total item-level revenue.

Repeat customers represent only **3.12%** of customers but contribute **5.73%** of revenue.

### Business Insight

Repeat customers contribute a larger share of revenue relative to their share of the customer base.

This suggests that repeat customers generate higher revenue per customer on average.

The next step is to quantify the average revenue per customer for each group and compare their purchasing behavior.

## 21. Average Revenue per Customer

### Business Question

How much revenue does the average one-time customer versus repeat customer generate?

### Objective

Compare total revenue with the number of customers in each group.

**Average Revenue per Customer = Customer Revenue ÷ Customer Count**9

### Result

Repeat customers generate an average revenue of **260.56 per customer**, compared with **138.62** for one-time customers.

This means the average revenue per repeat customer is approximately **1.88 times** that of a one-time customer.

### Business Insight

Repeat customers contribute more revenue per customer on average than one-time customers.

This reinforces the importance of understanding repeat-purchase behavior and customer retention. Further analysis should investigate how quickly customers make repeat purchases and which product categories are associated with repeat behavior.

## 22. Repeat Purchase Timing

### Business Question

How many days does it typically take for a customer to make a repeat purchase?

### SQL Approach

For each unique customer:

1. Orders are arranged chronologically.
2. `LAG()` retrieves the previous order date.
3. The difference between the current and previous order dates gives the repeat-purchase gap.
4. Only repeat purchases are included in the final calculation.

### Result

A total of **3,345 repeat-purchase intervals** were identified.

The average time between purchases is **77.86 days**, while the median is **28 days**.

The large difference between the mean and median indicates that some customers return much later than the typical repeat customer, increasing the average purchase gap.

### Business Insight

The median repeat-purchase gap of 28 days suggests that a meaningful portion of repeat purchases occurs relatively soon after the previous purchase.

The longer average gap indicates that some customers return after substantially longer periods.

This suggests that repeat-purchase behavior is heterogeneous and should be analyzed using purchase windows such as 30, 60, 90, and 180 days.

## 23. Repeat Purchase Windows

### Business Question

What percentage of repeat purchases occur within 30, 60, 90, and 180 days?

### Objective

Identify the time windows in which repeat purchases are most likely to occur.

This can help define appropriate customer re-engagement and retention windows.

### Result

The analysis identified 3,345 repeat-purchase intervals.

- 51.21% of repeat purchases occur within 30 days.
- 62.87% occur within 60 days.
- 69.63% occur within 90 days.
- 83.86% occur within 180 days.

### Business Insight

More than half of observed repeat purchases occur within 30 days, and approximately 70% occur within 90 days.

This indicates that the first 30–90 days after a purchase are important periods for understanding repeat-purchase behavior.

The remaining repeat purchases occur over longer intervals, showing that customer return behavior is not uniform.

These windows can be used in further retention and customer re-engagement analysis.

## 24. Customer Value Segmentation

### Business Question

How is customer revenue distributed across different customer-value segments?

### Approach

Customers are ranked based on their historical revenue.

Using `NTILE(4)`, customers are divided into four approximately equal-sized groups:

- Low Value
- Medium-Low
- Medium-High
- High Value

The revenue contribution of each segment is then calculated.

### Result

Customers were divided into four approximately equal-sized value segments using `NTILE(4)`.

Each segment contains approximately 25% of customers, but their revenue contributions differ substantially.

The High Value segment contains 23,855 customers and contributes **62.65%** of total historical customer revenue.

In comparison, the Low Value segment contains the same number of customers but contributes only **5.11%** of revenue.

### Business Insight

Customer revenue is highly concentrated within the High Value segment.

This demonstrates why customer-count analysis alone is insufficient for understanding business value. Customer segments should be evaluated using both customer population and revenue contribution.

The next step is to investigate the characteristics of these high-value customers, including order frequency, AOV, repeat behavior, and satisfaction.

## 25. Customer Order Frequency vs Revenue

### Business Question

How does customer order frequency relate to total customer revenue?

### Objective

Compare customers based on the number of orders they place and the total revenue they generate.

This helps determine whether higher purchase frequency is associated with higher customer value.

### Result

Customer revenue generally increases as order frequency increases.

Customers with one order generate an average revenue of **138.67**, while customers with two and three orders generate average revenues of **247.70** and **363.55**, respectively.

Higher-frequency groups show higher average revenue in general, although these groups contain very few customers and therefore should be interpreted cautiously.

### Business Insight

The analysis suggests a positive relationship between purchase frequency and customer revenue.

Repeat purchasing is therefore associated with higher customer value in this dataset. However, the relationship should be validated using additional metrics such as AOV and correlation analysis.

### Data Quality Note

Because customer revenue is calculated by joining `orders` with `order_items`, orders without matching item records are excluded from the revenue calculation. Therefore, customer-frequency results based only on orders may differ slightly from customer-frequency results involving item-level revenue.

## 26. Customer Order Frequency vs AOV

### Business Question

Does customer purchase frequency relate to Average Order Value (AOV)?

### Objective

Compare average order value across customers based on how many orders they have placed.

This helps determine whether high-frequency customers generate more value because they purchase more often, because they spend more per order, or both.

### Result

Customer order frequency does not show a consistent positive relationship with Average Order Value (AOV).

Customers with one order have an average AOV of **138.67**, while customers with two and three orders have lower average AOVs of **123.85** and **121.18**, respectively.

The higher-frequency groups contain very few customers, so their AOV values should be interpreted cautiously.

### Business Insight

The previous analysis showed that customers with more orders generally generate higher total revenue.

However, this analysis shows that higher customer value is not necessarily driven by higher AOV.

Instead, the results suggest that repeat customers can generate greater total revenue primarily through **purchase frequency**, rather than consistently spending more per individual order.

This distinction is important when designing customer-retention strategies.

## 27. Customer Value vs Delivery Time

### Business Question

Do higher-value customers experience different delivery times?

### Objective

Compare customer-level revenue with average delivery time to determine whether customer value is associated with delivery performance.

### Metrics

- Total customer revenue
- Number of orders
- Average delivery time

### Result

Average delivery time varies across customer-value segments.

Low Value customers have an average delivery time of **11.96 days**, while Very High Value customers have an average delivery time of **14.35 days**.

The observed delivery time increases across the customer-value segments.

### Business Insight

Higher-value customers do not appear to receive faster delivery in this analysis.

In fact, the Very High Value segment has the highest average delivery time among the four groups.

However, this is an observational relationship and does not establish that higher customer value causes slower delivery. Differences in product mix, geography, order characteristics, and other operational factors may contribute to the observed pattern.

## 28. Delivery Time vs Review Score

### Business Question

How is delivery time associated with customer review scores?

### Objective

Determine whether longer delivery times are associated with lower customer satisfaction.

### Approach

Orders are joined with order reviews using `order_id`.

Delivery time is grouped into ranges and compared with the average review score.

### Result

Average review score decreases as delivery time increases.

The average review score is:

- **4.45** for deliveries within 0–5 days
- **4.36** for 6–10 days
- **4.27** for 11–15 days
- **4.13** for 16–20 days
- **3.23** for deliveries taking more than 20 days

### Business Insight

There is a clear negative association between delivery time and customer review score.

Orders delivered within 0–5 days have an average review score of 4.45, while orders taking more than 20 days have an average score of 3.23.

This suggests that delivery performance is an important factor associated with customer satisfaction.

However, this analysis shows association rather than causation. Other factors such as product category, seller, location, and order characteristics may also influence review scores.

## 29. Review Score by Order Status

### Business Question

How does customer review score vary across different order statuses?

### Objective

Compare average customer review scores across order statuses to understand how order outcomes are associated with customer satisfaction.

### Result

Delivered orders have an average review score of **4.16** across 95,647 reviews.

Non-delivered order statuses generally have substantially lower average review scores. For example:

- Shipped: 2.01
- Canceled: 1.81
- Invoiced: 1.66
- Unavailable: 1.53
- Processing: 1.28

However, some statuses have very small review counts, such as approved (2 reviews) and created (3 reviews). Their average scores should therefore not be generalized.

### Business Insight

The large difference between delivered orders and major non-delivered statuses suggests that successful order completion is strongly associated with higher customer review scores.

This supports the importance of reliable order fulfillment and delivery performance for customer satisfaction.

However, this analysis shows association rather than causation.

## 30. Review Score Distribution

### Business Question

How are customer reviews distributed across the 1-to-5 star rating scale?

### Objective

Understand the overall distribution of customer satisfaction and identify the proportion of low- and high-rated reviews.

### Result

The review distribution is concentrated toward higher ratings.

- 5-star reviews account for **57.78%** of all reviews.
- 4-star and 5-star reviews together account for **77.07%**.
- 1-star and 2-star reviews together account for **14.69%**.

### Business Insight

Most reviews are positive, with 4- and 5-star ratings representing more than three-quarters of all reviews.

However, the presence of 1-star reviews indicates a meaningful group of dissatisfied customers. Further analysis of delivery performance and other order characteristics can help identify factors associated with lower review scores.

## 31. Overall Customer Satisfaction

### Business Question

What is the overall average customer review score?

### Objective

Calculate a single customer-satisfaction KPI that can be used as a high-level benchmark for the business.

### Result

The dataset contains **99,224 customer reviews**, with an overall average review score of **4.09 out of 5**.

### Business Insight

The overall review score indicates generally positive customer satisfaction.

However, the review distribution shows that a meaningful proportion of customers provide low ratings. Therefore, overall satisfaction should be analyzed together with delivery performance and order outcomes to identify areas contributing to dissatisfaction.

## 32. Delivery Time vs Review Score Correlation

### Business Question

What is the correlation between delivery time and customer review score?

### Objective

Quantify the strength and direction of the relationship between delivery time and customer satisfaction.

### Interpretation

- Correlation close to **-1** → strong negative relationship
- Correlation close to **0** → weak/no linear relationship
- Correlation close to **+1** → strong positive relationship

Correlation indicates association, not causation.

### Result

The analysis uses **96,359 order-review observations** and produces a delivery-time to review-score correlation of **-0.334**.

### Interpretation

The negative correlation indicates a moderate negative linear association between delivery time and review score.

In general, longer delivery times are associated with lower customer review scores.

This finding is consistent with the delivery-time bucket analysis, where average review scores decreased as delivery time increased.

### Business Insight

Delivery performance appears to be an important factor associated with customer satisfaction.

However, correlation does not establish causation. Other factors such as product category, seller, geography, and order characteristics may also influence review scores.

## 33. Customer Revenue vs Review Score Correlation

### Business Question

Is customer revenue associated with customer review scores?

### Objective

Measure whether customers who generate more revenue tend to provide higher or lower review scores.

Correlation is used to quantify the direction and strength of the relationship.

Correlation indicates association, not causation.

### Result

The analysis includes **94,721 customers** and produces a customer revenue to review score correlation of **-0.039**.

### Interpretation

The correlation is very close to zero, indicating a very weak negative linear association between customer revenue and average review score.

This suggests that higher-value customers do not systematically provide higher or lower review scores.

### Business Insight

Customer value appears to have little linear relationship with customer satisfaction in this analysis.

In comparison, delivery time showed a substantially stronger negative association with review score (-0.334).

This suggests that operational delivery performance may be more closely associated with customer satisfaction than customer revenue level.

However, correlation measures association and does not establish causation.

## 34. Order Frequency vs Customer Revenue Correlation

### Business Question

How strongly is customer order frequency associated with total customer revenue?

### Objective

Measure the relationship between the number of orders placed by a customer and their total historical revenue.

This helps quantify whether repeat purchasing is associated with higher customer value.

### Result

The analysis includes **95,420 customers** and produces a frequency-to-revenue correlation of **0.106**.

### Interpretation

The positive correlation indicates a weak positive linear association between customer order frequency and total customer revenue.

Customers with more orders generally tend to generate more total revenue, but the relationship is not strong enough to conclude that order frequency alone explains customer revenue differences.

### Business Insight

Purchase frequency is associated with higher customer revenue, but other factors also contribute to customer value.

Therefore, customer value analysis should combine multiple dimensions such as recency, frequency, monetary value, AOV, and product behavior.

## 35. RFM Customer Analysis

### Business Question

What are the Recency, Frequency, and Monetary characteristics of each customer?

### RFM Metrics

- **Recency:** Number of days since the customer's most recent purchase
- **Frequency:** Number of distinct orders placed by the customer
- **Monetary:** Total item-level revenue generated by the customer

### Objective

Create customer-level RFM metrics that can later be used for customer segmentation.

### Result

Customer-level Recency, Frequency, and Monetary (RFM) metrics were successfully calculated.

The results show that customer value can vary across different dimensions.

Some customers generate high monetary value despite having relatively few orders, while others have higher purchase frequency but lower total monetary value.

### Business Insight

RFM analysis provides a multidimensional view of customer behavior:

- Recency indicates how recently a customer purchased.
- Frequency indicates how often the customer purchases.
- Monetary indicates the customer's total historical revenue contribution.

These metrics can be combined to create meaningful customer segments in the next step.

## 36. RFM Scoring

### Business Question

How can customers be scored based on Recency, Frequency, and Monetary value?

### Scoring Method

Customers are divided into four quartiles using `NTILE(4)`.

#### Recency
More recent purchases receive a higher score.

#### Frequency
Customers with more orders receive a higher score.

#### Monetary
Customers with higher historical revenue receive a higher score.

The three scores can then be combined into an overall RFM score.

### Result

RFM scores were calculated using quartile-based scoring.

Each customer receives:

- Recency score: 1–4
- Frequency score: 1–4
- Monetary score: 1–4

The combined RFM score ranges from 3 to 12.

Customers with an RFM score of **12** are in the highest quartile for all three RFM dimensions under the selected scoring method.

### Important Interpretation Note

RFM scores are relative scores based on the distribution of customers.

A high Frequency score does not necessarily mean that a customer has a large absolute number of orders. It means the customer's frequency falls within a higher quartile of the customer distribution.

## 37. RFM Customer Segmentation

### Business Question

How can customers be grouped into actionable RFM segments?

### Segment Logic

Customers are grouped using their combined RFM score:

- **Champions:** RFM score 10–12
- **Loyal Customers:** RFM score 7–9
- **Potential Customers:** RFM score 5–6
- **At Risk / Low Value:** RFM score 3–4

These labels provide a business-friendly interpretation of the RFM scores.

### Result

RFM scoring was used to classify 95,420 customers into four business-oriented segments.

- Loyal Customers: 37.57%
- Potential Customers: 25.30%
- Champions: 24.88%
- At Risk / Low Value: 12.25%

Loyal Customers represent the largest segment.

Together, Loyal Customers and Champions account for 62.45% of analyzed customers.

### Business Insight

The RFM segmentation provides a customer-level view of behavioral value.

However, customer share alone does not indicate business contribution. The next step is to compare revenue contribution across RFM segments to determine which segments generate the largest share of total revenue.

## 38. RFM Segment Revenue Contribution

### Business Question

How much revenue does each RFM customer segment contribute?

### Objective

Compare customer count and revenue contribution across RFM segments.

This helps identify which customer segments are most valuable from a revenue perspective.

### Result

RFM segments show a strong difference between customer share and revenue contribution.

Champions represent **24.88%** of customers but contribute **48.10%** of total revenue.

Loyal Customers represent **37.57%** of customers and contribute **40.93%** of total revenue.

Together, Champions and Loyal Customers represent **62.45% of customers** and contribute **89.03% of revenue**.

Potential Customers account for **25.30% of customers** but contribute **8.49% of revenue**.

At Risk / Low Value customers account for **12.25% of customers** but contribute only **2.48% of revenue**.

### Business Insight

Revenue is highly concentrated among the higher-scoring RFM segments.

This demonstrates that customer count and customer value are not proportional. RFM segmentation can therefore help prioritize customer analysis by combining recency, purchase frequency, and historical monetary value.

The results should be interpreted within the chosen RFM scoring methodology and historical dataset period.

## 39. RFM Segment Average Order Value

### Business Question

How does Average Order Value (AOV) differ across RFM customer segments?

### Objective

Compare the average order value of each RFM segment to understand whether higher-value customer segments also generate larger orders.

This helps distinguish the effect of purchase frequency from per-order spending.

### Result

Average Order Value varies substantially across RFM segments.

Champions have the highest average order value at **270.54**, followed by Loyal Customers at **138.68**.

Potential Customers have an average order value of **46.84**, while At Risk / Low Value customers have the lowest average order value at **29.09**.

### Business Insight

Champions generate substantially higher value per order than the other RFM segments.

This indicates that their higher monetary contribution is associated not only with customer purchasing behavior but also with substantially higher average order values.

The results reinforce the importance of analyzing both purchase frequency and order value when evaluating customer value.

### Data Consistency Note

RFM segment counts can vary slightly when different order-item joins or customer populations are used. Final reporting should use a consistent RFM customer base across all segment-level metrics.

## 40. RFM Segment vs Customer Satisfaction

### Business Question

How does customer satisfaction vary across RFM segments?

### Objective

Compare average review scores across RFM segments to determine whether higher-value customer segments have different satisfaction levels.

Correlation or segment comparison indicates association, not causation.

### Result

Average review scores are relatively similar across all RFM segments.

At Risk / Low Value customers have the highest average review score at **4.16**, while Champions have an average score of **4.07**.

The difference between the highest and lowest segment averages is only **0.09 points**.

### Business Insight

RFM customer value does not appear to have a strong relationship with customer satisfaction in this analysis.

Higher-value customers do not necessarily provide higher review scores. This is consistent with the very weak customer revenue-to-review correlation observed earlier.

Therefore, customer value and customer satisfaction should be treated as separate dimensions when analyzing customer behavior.

### Data Interpretation Note

The analysis shows association within the observed dataset and does not establish causation.

## 41. RFM Segment vs Delivery Performance

### Business Question

How does average delivery time vary across RFM customer segments?

### Objective

Compare delivery performance across customer-value segments to determine whether higher-value customers experience different delivery times.

This analysis measures association between customer value and delivery performance.

### Result

Average delivery time varies across RFM segments.

Potential Customers have the lowest average delivery time at **11.81 days**, while Loyal Customers have the highest at **13.09 days**.

Champions have an average delivery time of **12.63 days**.

### Business Insight

Higher-value RFM segments do not appear to receive systematically faster delivery.

In fact, Champions and Loyal Customers have higher average delivery times than Potential and At Risk / Low Value customers in this analysis.

This suggests that customer value is not clearly associated with faster delivery performance.

However, this is an observational comparison. Differences in geography, product category, seller, and order characteristics may influence delivery time.

## 42. RFM Segment Business Summary

### Business Question

What are the key business characteristics of each RFM customer segment?

### Objective

Create a consolidated customer-segment view containing:

- Customer count
- Total revenue
- Revenue share
- Average order value
- Average review score
- Average delivery time

This provides a single business summary of customer segments.

### Final RFM Business Insights

The consolidated RFM analysis shows a strong concentration of revenue among higher-value customer segments.

Champions represent approximately 25% of the customer base but contribute **49.86% of total revenue**. Together, Champions and Loyal Customers represent approximately **63.16% of customers and 88.93% of revenue**.

Average Order Value also varies substantially across segments. Champions have an average order value of **270.61**, compared with **29.45** for At Risk / Low Value customers.

However, customer satisfaction does not increase with customer value. Average review scores range from 4.06 to 4.16 across the segments, indicating relatively small differences.

Similarly, higher-value customers do not consistently receive faster delivery. Champions average 12.72 delivery days, while Potential Customers average 11.87 days.

Overall, RFM analysis shows that customer value is strongly differentiated by purchasing behavior and order value, while satisfaction and delivery performance appear to be relatively independent dimensions.

## 43. Product Category × Customer Value

### Business Question

Which product categories generate the most revenue from higher-value customers?

### Objective

Analyze revenue by product category and RFM customer segment.

This helps identify categories that are particularly important among Champions, Loyal Customers, Potential Customers, and At Risk / Low Value customers.

### Business Use

The analysis can help identify:
- Categories preferred by high-value customers
- Categories with strong revenue contribution
- Opportunities for targeted customer campaigns

### Initial Observation

The analysis successfully links product categories with RFM customer segments.

The output shows that category-level revenue differs across customer segments. The complete result will be used to identify the categories contributing the most revenue within each RFM segment.

### Result

The category analysis shows that product-category performance varies across RFM customer segments.

Among Champions, `relogios_presentes`, `beleza_saude`, and `esporte_lazer` are the top revenue-generating categories.

Among Loyal Customers, `cama_mesa_banho`, `beleza_saude`, and `esporte_lazer` generate the highest revenue.

Potential Customers show a similar pattern, with `cama_mesa_banho`, `beleza_saude`, and `esporte_lazer` leading their category revenue.

The At Risk / Low Value segment has `telefonia`, `informatica_acessorios`, and `moveis_decoracao` as its top categories.

### Business Insight

Several categories appear consistently across higher-value customer segments, particularly `beleza_saude` and `esporte_lazer`.

However, category performance differs by customer segment. Therefore, category strategy should consider both product revenue and customer-value segment rather than evaluating categories only on overall revenue.

## 44. Product Category × Repeat Customer Rate

### Business Question

Which product categories have the highest proportion of repeat customers?

### Objective

Compare customer retention behavior across product categories.

A category may generate high revenue but have a low repeat-customer rate. Therefore, both revenue and repeat behavior should be considered when evaluating category performance.

### Result

Repeat-customer rates vary substantially across product categories.

Some smaller categories show relatively high repeat-customer rates, such as `livros_importados` (13.46%) and `fashion_roupa_masculina` (10.91%). However, these categories have relatively small customer populations.

`cama_mesa_banho` has a lower repeat rate of 8.90%, but it has a much larger customer base of 9,145 customers and 814 repeat customers.

### Business Insight

Repeat-customer rate should be interpreted together with the size of the customer base.

A category with a high percentage but a very small customer population may contribute fewer repeat customers than a larger category with a slightly lower percentage.

The analysis also shows that several high-revenue categories have relatively low repeat-customer rates. This suggests that category revenue and customer retention are separate dimensions and should be evaluated together.

## 45. Category Revenue vs Repeat Customer Rate

### Business Question

Which product categories combine strong revenue contribution with repeat-customer behavior?

### Objective

Compare category revenue with repeat-customer rate.

This helps distinguish:
- High-revenue categories
- High-retention categories
- Categories that perform strongly on both dimensions

### Business Insight

Revenue and retention represent different dimensions of category performance. A category can generate high revenue without having a high repeat-customer rate, or have a high repeat rate with a relatively small customer base.

### Result

The combined category analysis shows that revenue and repeat-customer behavior vary independently across product categories.

`beleza_saude` generates the highest revenue among the analyzed categories at approximately **1.26 million**, but its repeat-customer rate is only **1.74%**.

`cama_mesa_banho` generates approximately **1.04 million** in revenue and has the highest repeat-customer rate among the top revenue categories at **2.76%**.

`esporte_lazer` also combines substantial revenue (**988K**) with a relatively higher repeat-customer rate of **2.45%**.

In contrast, `cool_stuff` generates approximately **635K** in revenue but has a repeat-customer rate of only **0.47%**.

### Business Insight

High revenue and customer retention are separate dimensions of category performance.

Categories with strong revenue but low repeat rates may represent opportunities for customer retention and re-engagement initiatives.

Therefore, category performance should be evaluated using both revenue contribution and repeat-customer behavior rather than revenue alone.

## 46. Order Frequency vs Average Order Value

### Business Question

Is customer purchase frequency associated with Average Order Value (AOV)?

### Objective

Measure the relationship between the number of orders placed by a customer and their average order value.

This helps determine whether frequent customers also tend to place larger orders.

### Interpretation

Correlation measures the strength of a linear association. It does not establish causation.

### Result

The analysis includes **95,420 customers** and produces a frequency-to-AOV correlation of **-0.011**.

### Interpretation

The correlation is extremely close to zero, indicating virtually no linear relationship between customer order frequency and average order value.

Customers who purchase more frequently do not necessarily place larger individual orders.

### Business Insight

The earlier frequency-to-revenue analysis showed a weak positive relationship (0.106), while frequency-to-AOV shows almost no relationship (-0.011).

This suggests that higher revenue among frequent customers is more closely associated with having more purchase occasions rather than consistently higher order values.

## 47. Customer Revenue vs Average Order Value

### Business Question

Is total customer revenue associated with Average Order Value?

### Objective

Measure the relationship between a customer's historical revenue and their average order value.

This helps determine whether high-value customers generate more revenue primarily because of larger orders or other purchasing behavior.

### Interpretation

Correlation measures association and does not establish causation.

### Result

The analysis includes **95,420 customers** and produces a revenue-to-AOV correlation of **0.985**.

### Interpretation

The very strong positive correlation indicates that customers with higher total historical revenue generally also have higher average order values.

However, total revenue is mathematically related to both purchase frequency and order value. Therefore, the strong revenue-to-AOV relationship should not be interpreted as an independent causal relationship.

### Combined Insight

The correlation analysis shows:

- Frequency vs Revenue: **0.106** — weak positive association
- Frequency vs AOV: **-0.011** — virtually no linear association
- Revenue vs AOV: **0.985** — very strong positive association

This indicates that customer monetary value is strongly associated with order value, while purchase frequency alone has a much weaker linear relationship with total revenue.

### Result

The analysis uses **96,359 order-review observations** and produces a delivery-time to review-score correlation of **-0.334**.

### Interpretation

The negative correlation indicates a moderate negative association between delivery time and review score.

Longer delivery times are generally associated with lower customer review scores.

This result is also consistent with the delivery-time bucket analysis, where average review scores decreased as delivery time increased.

### Business Insight

Delivery performance appears to be an important dimension associated with customer satisfaction.

However, correlation does not establish causation. Other factors such as product category, seller, geography, and order characteristics may also influence review scores.

## 49. Customer Revenue vs Delivery Time

### Business Question

Is customer revenue associated with delivery time?

### Objective

Measure whether customers generating higher historical revenue experience systematically different delivery times.

This complements the RFM segment-level delivery analysis.

### Interpretation

Correlation measures association and does not establish causation.

### Result

The analysis includes **93,356 customers** and produces a revenue-to-delivery-time correlation of **0.055**.

### Interpretation

The correlation is very close to zero, indicating a very weak positive linear association between customer revenue and average delivery time.

Higher-revenue customers do not appear to systematically experience faster or slower delivery.

### Business Insight

Customer revenue has little linear relationship with delivery time.

This is consistent with the RFM segment analysis, where higher-value customers did not consistently receive faster delivery.

Therefore, delivery performance appears to be relatively independent of customer monetary value in this dataset.

Correlation indicates association and does not establish causation.

## 50. Final SQL Business KPI Summary

### Objective

Create a consolidated view of the key business KPIs identified throughout the SQL analysis.

The summary includes revenue, orders, customers, AOV, repeat-customer rate, delivery performance, review score, payment mix, and the top revenue-generating category.

This provides a final executive-level snapshot of the RetailIQ business.

### Final SQL KPI Snapshot

The consolidated SQL analysis produces the following business KPIs:

- Total Revenue: **13.59 million**
- Total Orders: **99,441**
- Overall AOV: **136.68**
- Repeat Customer Rate: **3.12%**
- Average Delivery Time: **12.56 days**
- Average Review Score: **4.09/5**
- Credit Card Revenue Share: **78.34%**
- Top Revenue Category: **beleza_saude**
- Top Category Revenue: **1.26 million**

These KPIs provide an executive-level summary of the historical e-commerce business performance.

Customer populations may vary slightly across analyses because different metrics use different data-join requirements. Final reporting should maintain consistent metric definitions.

## 51. Final Database Validation

### Objective

Perform final integrity checks before completing the SQL analysis phase.

The validation checks:

- Primary-key uniqueness
- Foreign-key relationships
- Expected row counts
- Missing key values

A successful validation confirms that the database is ready for downstream BI and dashboard development.